In [4]:
%pip install kagglehub

  Using cached kagglehub-1.0.2-py3-none-any.whl (70 kB)
  Using cached kagglesdk-0.1.28-py3-none-any.whl (230 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import kagglehub

dataset_path = kagglehub.dataset_download("soumikrakshit/anime-faces")
print("Dataset downloaded to:", dataset_path)

c:\Users\MINA\Documents\WeekilyPrac\Ml_JOUNRAL\ml-learning-journal\GEN AI\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 441M/441M [01:38<00:00, 4.69MB/s] 

Extracting files...


Dataset downloaded to: C:\Users\MINA\.cache\kagglehub\datasets\soumikrakshit\anime-faces\versions\1


In [3]:
import sys
print(sys.executable)

c:\Users\MINA\Documents\WeekilyPrac\Ml_JOUNRAL\ml-learning-journal\GEN AI\.venv\Scripts\python.exe


In [8]:
from PIL import Image
import glob
import os
import numpy as np

IMG_SIZE = 64

image_path = glob.glob(os.path.join(dataset_path, "**", "*png"), recursive=True)
image_path += glob.glob(os.path.join(dataset_path, "**", "*jpg"), recursive=True) 

print(f"i found {len(image_path)} images")

images = []

for path in image_path:
    img = Image.open(path).convert('RGB').resize((IMG_SIZE,IMG_SIZE))
    images.append(np.array(img))

X_train = np.array(images, dtype='float32')/255.0

print(X_train.shape)

i found 43102 images
(43102, 64, 64, 3)


In [11]:
import glob 
import os
import numpy as np
from PIL import Image 

image_path = glob.glob(os.path.join(dataset_path, "**", "*.png"), recursive=True)
image_path += glob.glob(os.path.join(dataset_path, "**", "*.jpeg"), recursive=True)

IMG_SIZE = 64
images = []

for path in image_path:
    img = Image.open(path).convert('RGB').resize((IMG_SIZE,IMG_SIZE))
    images.append(np.array(img))
X_train = np.array(images, dtype='float32')/255.0 
print(X_train.shape)

(43102, 64, 64, 3)


### Generator

In [12]:
import tensorflow as tf 
from tensorflow import keras 
from tensorflow.keras import layers 

In [20]:

128/2


64.0

In [ ]:
# 64 - 32 - 16 - 8 
# 256- 128 - 64 - 3

In [23]:
latend_dim = 100 

generator = keras.Sequential([
    layers.Dense(8*8*256, input_dim=(latend_dim)),
    layers.BatchNormalization(),
    layers.LeakyReLU(),

    layers.Reshape((8,8,256)),

    layers.Conv2DTranspose(128, kernel_size=4, strides=2, padding='same'),
    layers.BatchNormalization(),
    layers.LeakyReLU(),

    layers.Conv2DTranspose(64, kernel_size=4, strides=2, padding='same'),
    layers.BatchNormalization(),
    layers.LeakyReLU(),

    layers.Conv2DTranspose(3, kernel_size=4, strides=2, padding='same')

])

generator.summary()

Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_9 (Dense)             (None, 16384)             1654784   
                                                                 
 batch_normalization_12 (Bat  (None, 16384)            65536     
 chNormalization)                                                
                                                                 
 leaky_re_lu_12 (LeakyReLU)  (None, 16384)             0         
                                                                 
 reshape_4 (Reshape)         (None, 8, 8, 256)         0         
                                                                 
 conv2d_transpose_11 (Conv2D  (None, 16, 16, 128)      524416    
 Transpose)                                                      
                                                                 
 batch_normalization_13 (Bat  (None, 16, 16, 128)     

In [27]:
discriminator = keras.Sequential([
    layers.Input(shape=(64,64,3)),

    layers.Conv2D(64, kernel_size=4, strides=2, padding='same'),
    layers.LeakyReLU(),
    layers.Dropout(0.3),

    layers.Conv2D(128, kernel_size=4, strides=2, padding='same'),
    layers.LeakyReLU(),
    layers.Dropout(0.3),

    layers.Conv2D(256, kernel_size=4, strides=2, padding='same'),
    layers.LeakyReLU(),
    layers.Dropout(0.3),

    layers.Flatten(),
    layers.Dense(1, activation='sigmoid')
])

discriminator.summary()

Model: "sequential_7"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_6 (Conv2D)           (None, 32, 32, 64)        3136      
                                                                 
 leaky_re_lu_21 (LeakyReLU)  (None, 32, 32, 64)        0         
                                                                 
 dropout_6 (Dropout)         (None, 32, 32, 64)        0         
                                                                 
 conv2d_7 (Conv2D)           (None, 16, 16, 128)       131200    
                                                                 
 leaky_re_lu_22 (LeakyReLU)  (None, 16, 16, 128)       0         
                                                                 
 dropout_7 (Dropout)         (None, 16, 16, 128)       0         
                                                                 
 conv2d_8 (Conv2D)           (None, 8, 8, 256)        

In [29]:
discriminator.compile(optimizer = keras.optimizers.Adam(learning_rate=0.0002), loss='binary_crossentropy', metrics=['accuracy'])

In [30]:
gan_input = keras.Input(shape=(latend_dim))
fake_image = generator(gan_input)
gan_output = discriminator(fake_image)

gan = keras.Model(gan_input, gan_output)

gan.compile(optimizer = keras.optimizers.Adam(learning_rate=0.0002), loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
epochs = 5000 
batch_size=128 

for epoch in range(epochs):
    idx = np.random.randint(0,X_train.shape[0],batch_size)
    real_images = X_train[idx]

    noise = np.random.normal(0,1, (batch_size,latend_dim))
    fake_images = generator.predict(noise, verbose=0)

    d_loss_real = discriminator.train_on_batch(real_images, np.ones((batch_size,1)))
    d_loss_fake = discriminator.train_on_batch(fake_images, np.zeros((batch_size,1)))

    noise = np.random.normal(0,1, (batch_size,latend_dim))
    g_loss = gan.train_on_batch(noise, np.ones((batch_size,1)))

    if epoch % 1000 == 0:
        print()

In [ ]:
noise = np.random.normal(0,1, (10,latend_dim))

generated_image = generator.predict(noise, verbose=0)